In [1]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT_DIR = "../data_ready_for_kaggle"
TEST_PATH = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR = os.path.abspath("../saved_models_v3")
RESULTS_DIR = "./results"
MAX_TEST_IMAGES = 200
MAX_LENGTH = 64
BATCH_SIZE = 16
NUM_WORKERS = 0
NUM_BEAMS = 5
NO_REPEAT_NGRAM_SIZE = 3
REPETITION_PENALTY = 1.5
WARMUP_STEPS = 10
device = torch.device("cpu")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Device: {device}")
print(f"Model dir: {MODEL_DIR}")


Device: cpu
Model dir: c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\saved_models_v3


In [3]:
class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=64):
        self.df = pd.read_csv(data_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_length = max_length

        self.image_groups = (
            self.df.groupby('image_file')['caption']
            .apply(list)
            .reset_index()
        )

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, index):
        row = self.image_groups.iloc[index]
        image_file = row['image_file']
        caption = row['caption'][0]

        image_path = os.path.join(self.img_dir, image_file)
        image = Image.open(image_path).convert('RGB')

        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding['image_file'] = image_file
        encoding['raw_captions'] = row['caption']
        return encoding

def collate_fn(batch):
    tensor_keys = ['pixel_values', 'input_ids', 'attention_mask']
    result = {}
    for key in tensor_keys:
        result[key] = torch.stack([item[key] for item in batch])
    result['image_file'] = [item['image_file'] for item in batch]
    result['raw_captions'] = [item['raw_captions'] for item in batch]
    return result


In [4]:
processor = BlipProcessor.from_pretrained(MODEL_DIR)
test_dataset = UITViICDataset(TEST_PATH, IMAGES_DIR, processor, max_length=MAX_LENGTH)
test_dataset.image_groups = test_dataset.image_groups.head(MAX_TEST_IMAGES).reset_index(drop=True)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

print(f"Test images: {len(test_dataset)}")
print(f"Test batches: {len(test_dataloader)}")


Test images: 200
Test batches: 13


In [5]:
def build_multi_references(dataloader):
    references = {}
    for batch in dataloader:
        for image_file, raw_captions in zip(batch['image_file'], batch['raw_captions']):
            references[image_file] = raw_captions
    return references

def generate_captions(model, processor, dataloader, device):
    model.eval()
    results = {}
    latencies = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Generating")):
            pixel_values = batch['pixel_values'].to(device)
            image_files = batch['image_file']

            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = time.perf_counter()

            generated_ids = model.generate(
                pixel_values=pixel_values,
                max_length=MAX_LENGTH,
                num_beams=NUM_BEAMS,
                early_stopping=True,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                repetition_penalty=REPETITION_PENALTY,
            )

            if device.type == 'cuda':
                torch.cuda.synchronize()
            end_time = time.perf_counter()

            if batch_idx >= WARMUP_STEPS:
                latencies.append(end_time - start_time)

            preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
            for image_file, pred in zip(image_files, preds):
                results[image_file] = pred

    return results, latencies

def calculate_metrics(predictions_dict, references_dict):
    common_images = sorted(set(predictions_dict.keys()) & set(references_dict.keys()))
    predictions = [predictions_dict[img] for img in common_images]
    references = [references_dict[img] for img in common_images]

    bleu_metric = evaluate.load('bleu')
    rouge_metric = evaluate.load('rouge')
    meteor_metric = evaluate.load('meteor')

    bleu_results = bleu_metric.compute(predictions=predictions, references=references)
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    meteor_results = meteor_metric.compute(predictions=predictions, references=references)

    return {
        'bleu': bleu_results,
        'rouge': rouge_results,
        'meteor': meteor_results,
        'num_images': len(common_images),
    }

def summarize_benchmark(name, predictions_dict, references_dict, latencies, backend, precision, device, provider=None):
    metrics = calculate_metrics(predictions_dict, references_dict)

    avg_latency = float(np.mean(latencies)) if latencies else 0.0
    p95_latency = float(np.percentile(latencies, 95)) if latencies else 0.0
    throughput = float(BATCH_SIZE / avg_latency) if avg_latency > 0 else 0.0

    benchmark = {
        'name': name,
        'backend': backend,
        'precision': precision,
        'device': device,
        'provider': provider,
        'batch_size': BATCH_SIZE,
        'max_test_images': MAX_TEST_IMAGES,
        'avg_latency_seconds_per_batch': avg_latency,
        'p95_latency_seconds_per_batch': p95_latency,
        'throughput_images_per_second': throughput,
        'bleu4': metrics['bleu']['bleu'] * 100,
        'rougeL': metrics['rouge']['rougeL'] * 100,
        'meteor': metrics['meteor']['meteor'] * 100,
        'num_images': metrics['num_images'],
    }
    return benchmark, metrics


In [6]:
model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(device)
references_dict = build_multi_references(test_dataloader)
predictions_dict, latencies = generate_captions(model, processor, test_dataloader, device)
benchmark, metrics = summarize_benchmark(
    'baseline_fp32',
    predictions_dict,
    references_dict,
    latencies,
    backend='pytorch',
    precision='fp32',
    device=device.type,
    provider='torch'
)

print(json.dumps(benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'baseline_fp32.json'), 'w', encoding='utf-8') as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)

with open(os.path.join(RESULTS_DIR, 'baseline_fp32_predictions.json'), 'w', encoding='utf-8') as f:
    json.dump(predictions_dict, f, ensure_ascii=False, indent=2)


Generating: 100%|██████████| 13/13 [07:19<00:00, 33.81s/it]
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nviquang\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


{
  "name": "baseline_fp32",
  "backend": "pytorch",
  "precision": "fp32",
  "device": "cpu",
  "provider": "torch",
  "batch_size": 16,
  "max_test_images": 200,
  "avg_latency_seconds_per_batch": 28.356538733331643,
  "p95_latency_seconds_per_batch": 35.28962972999871,
  "throughput_images_per_second": 0.5642437587487653,
  "bleu4": 24.503374917356012,
  "rougeL": 50.55289099627949,
  "meteor": 35.88133955254793,
  "num_images": 200
}
